# S-N Curve Library — 221 Curves Across 17 Standards
### digitalmodel Fatigue Engineering Toolkit

This notebook demonstrates ACE Engineer's comprehensive S-N curve library covering:
- **17 international standards** (DNV, BS, API, EC3, IIW, and more)
- **221 curves** spanning weld classes, environments, and detail categories
- Programmatic access via clean API for automated fatigue assessment

**Standards covered**: DNV-RP-C203, API RP 2A, BS 7608, IIW, Eurocode 3, NORSOK N-004, AWS D1.1, ASME VIII, EN 13445, PD 5500, ABS, ISO 19902, CSA S16, HSE OTH 92/390, DoE Guidance, GL Rules, JIS B 8266

**Key capabilities**:
1. Browse and filter the full curve catalogue
2. Compare curves across standards for the same detail category
3. Evaluate environment effects (air / cathodic protection / free corrosion)
4. Calculate fatigue endurance for any stress range
5. SCF calculation for tubular joints and plated connections
6. Hotspot stress extrapolation per DNV-RP-C203 and IIW

In [ ]:
"""Cell 2: Catalog Overview — Browse the full S-N curve library."""

from digitalmodel.fatigue.sn_library_api import (
    list_curves, get_curve, calculate_endurance, compare_curves,
)
import numpy as np
import pandas as pd

# List every curve in the library
all_curves = list_curves()
print(f"Total curves in library: {len(all_curves)}")

# Build a DataFrame for analysis
df = pd.DataFrame([c.model_dump() for c in all_curves])

# Curves per standard
print("\n--- Curves by Standard ---")
print(df.groupby("standard").size().sort_values(ascending=False).to_string())

# Environment breakdown
print(f"\nEnvironments: {df['environment'].unique().tolist()}")

# Weld class sample
classes = sorted(df["weld_class"].unique())
print(f"Weld classes ({len(classes)} unique): {classes[:20]} ...")

# Slope distribution
print(f"\nSlope m1 range: {df['m1'].min():.1f} – {df['m1'].max():.1f}")
print(f"Curves with endurance limit: {df['endurance_limit'].notna().sum()} / {len(df)}")

In [ ]:
"""Cell 3: Standard Comparison — DNV vs BS vs API for class 'D' in air."""

import plotly.graph_objects as go

# Compare the 'D' class across three major offshore/structural standards
curve_ids = ["DNV-RP-C203:D:air", "BS 7608:D:air", "API RP 2A:D:air"]
stress_ranges = np.logspace(np.log10(10), np.log10(500), 200)

comparison = compare_curves(curve_ids, stress_ranges)

fig = go.Figure()
colors = ["#1f77b4", "#d62728", "#2ca02c"]
for i, name in enumerate(comparison.curve_names):
    cycles = np.array(comparison.endurance_cycles[i])
    fig.add_trace(go.Scatter(
        x=cycles, y=stress_ranges,
        mode="lines", name=name,
        line=dict(width=2.5, color=colors[i]),
        hovertemplate="N = %{x:.2e}<br>S = %{y:.1f} MPa<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(
    title="S-N Curve Comparison — Class D, Air Environment",
    xaxis=dict(title="Cycles to Failure (N)", type="log", showgrid=True, gridcolor="#eee"),
    yaxis=dict(title="Stress Range (MPa)", type="log", showgrid=True, gridcolor="#eee"),
    template="plotly_white",
    width=900, height=550,
    legend=dict(x=0.02, y=0.02, bgcolor="rgba(255,255,255,0.8)"),
)
fig.show()

# Print key parameters side by side
print("\n--- Curve Parameters ---")
for cid in curve_ids:
    c = get_curve(cid)
    print(f"\n{c.curve_id}:")
    print(f"  m1={c.m1}, log_a1={c.log_a1:.4f}, m2={c.m2}, log_a2={c.log_a2}")
    print(f"  Knee at N={c.knee_point:.0e}, CAFL={c.endurance_limit} MPa")

In [ ]:
"""Cell 4: Environment Effects — Air vs Seawater CP vs Free Corrosion."""

# DNV-RP-C203 class D across all three environments
env_curves = [
    "DNV-RP-C203:D:air",
    "DNV-RP-C203:D:seawater_cp",
    "DNV-RP-C203:D:free_corrosion",
]
env_labels = ["Air", "Seawater with CP", "Free Corrosion"]
env_colors = ["#2ca02c", "#1f77b4", "#d62728"]

stress_ranges = np.logspace(np.log10(10), np.log10(400), 200)
env_comparison = compare_curves(env_curves, stress_ranges)

fig_env = go.Figure()
for i, (name, label) in enumerate(zip(env_comparison.curve_names, env_labels)):
    cycles = np.array(env_comparison.endurance_cycles[i])
    fig_env.add_trace(go.Scatter(
        x=cycles, y=stress_ranges,
        mode="lines", name=label,
        line=dict(width=2.5, color=env_colors[i]),
        hovertemplate="N = %{x:.2e}<br>S = %{y:.1f} MPa<extra>%{fullData.name}</extra>",
    ))

fig_env.update_layout(
    title="Environment Effect on Fatigue Life — DNV-RP-C203 Class D",
    xaxis=dict(title="Cycles to Failure (N)", type="log", showgrid=True, gridcolor="#eee"),
    yaxis=dict(title="Stress Range (MPa)", type="log", showgrid=True, gridcolor="#eee"),
    template="plotly_white",
    width=900, height=550,
    legend=dict(x=0.02, y=0.02, bgcolor="rgba(255,255,255,0.8)"),
)
fig_env.show()

# Quantify the penalty: fatigue life at 100 MPa stress range
print("\n--- Fatigue Life at 100 MPa Stress Range ---")
for cid, label in zip(env_curves, env_labels):
    c = get_curve(cid)
    N = calculate_endurance(c, 100.0)
    print(f"  {label:25s}: N = {N:.2e} cycles")

# Life reduction factors
c_air = get_curve(env_curves[0])
N_air = calculate_endurance(c_air, 100.0)
for cid, label in zip(env_curves[1:], env_labels[1:]):
    c = get_curve(cid)
    N = calculate_endurance(c, 100.0)
    print(f"  Reduction ({label} vs Air): {N/N_air:.2f}x")

In [ ]:
"""Cell 5: Fatigue Life Calculation — Miner's Rule Worked Example.

Scenario: A welded tubular joint (DNV class D, seawater with CP) subjected
to a wave-induced stress histogram. We calculate cumulative fatigue damage
and remaining design life.
"""

# --- Define the stress histogram (typical 20-year wave loading) ---
# Each bin: (stress_range_MPa, number_of_cycles_in_service)
stress_histogram = [
    (200.0,   5_000),
    (175.0,  15_000),
    (150.0,  50_000),
    (125.0, 200_000),
    (100.0, 800_000),
    ( 75.0, 2_000_000),
    ( 50.0, 5_000_000),
    ( 30.0, 15_000_000),
    ( 20.0, 40_000_000),
]

# --- Select S-N curve ---
curve = get_curve("DNV-RP-C203:D:seawater_cp")
print(f"S-N Curve: {curve.curve_id}")
print(f"  Slope m1={curve.m1}, log_a1={curve.log_a1:.4f}")
print(f"  Slope m2={curve.m2}, log_a2={curve.log_a2}")
print(f"  Knee at N={curve.knee_point:.0e}")

# --- Palmgren-Miner summation ---
print(f"\n{'Stress (MPa)':>14s} {'n_applied':>12s} {'N_allow':>12s} {'Damage n/N':>12s}")
print("-" * 54)

total_damage = 0.0
rows = []
for S, n in stress_histogram:
    N = calculate_endurance(curve, S)
    damage = n / N
    total_damage += damage
    rows.append({"S_range_MPa": S, "n_applied": n, "N_allowable": N, "damage": damage})
    print(f"{S:14.1f} {n:12,d} {N:12.2e} {damage:12.6f}")

print("-" * 54)
print(f"{'Cumulative Miner damage':>40s}: {total_damage:.4f}")

# --- Design life ---
service_years = 20.0
DFF = 3.0  # Design Fatigue Factor per DNV-RP-C203 Table 8-1
print(f"\nService period: {service_years:.0f} years")
print(f"Design Fatigue Factor (DFF): {DFF:.1f}")
print(f"Required damage limit: 1/DFF = {1/DFF:.4f}")
print(f"Calculated fatigue life: {service_years / total_damage:.1f} years")
print(f"Design fatigue life (with DFF): {service_years / (total_damage * DFF):.1f} years")

if total_damage * DFF < 1.0:
    print("\n>> PASS: Fatigue damage within allowable limit.")
else:
    print("\n>> FAIL: Fatigue damage exceeds allowable — redesign required.")

In [ ]:
"""Cell 6: SCF + Hotspot Stress Workflow.

End-to-end: tubular joint geometry -> SCF -> hotspot stress -> fatigue damage.
This demonstrates how the fatigue toolkit modules work together.
"""

from digitalmodel.fatigue.scf_library import (
    TubularJointGeometry, efthymiou_ty_axial, efthymiou_ty_ipb, efthymiou_ty_opb,
)
from digitalmodel.fatigue.hotspot_stress import (
    extrapolate_hotspot_linear, recommended_readout_distances,
)

# --- Step 1: Define tubular joint geometry ---
joint = TubularJointGeometry(
    D=1200.0,   # chord OD (mm)
    T=50.0,     # chord wall thickness (mm)
    d=600.0,    # brace OD (mm)
    t=25.0,     # brace wall thickness (mm)
    theta=45.0, # brace angle (degrees)
)

print("=== Tubular Joint Geometry ===")
print(f"  Chord: D={joint.D}mm, T={joint.T}mm")
print(f"  Brace: d={joint.d}mm, t={joint.t}mm, theta={joint.theta} deg")
print(f"  beta={joint.beta:.3f}, gamma={joint.gamma:.1f}, tau={joint.tau:.2f}")

# --- Step 2: Calculate SCFs for each load type ---
scf_ax = efthymiou_ty_axial(joint)
scf_ipb = efthymiou_ty_ipb(joint)
scf_opb = efthymiou_ty_opb(joint)

print("\n=== Stress Concentration Factors (Efthymiou) ===")
for label, scf in [("Axial", scf_ax), ("In-plane bending", scf_ipb), ("Out-of-plane bending", scf_opb)]:
    print(f"  {label:25s}: chord={scf.scf_chord:.2f}, brace={scf.scf_brace:.2f}, governing={scf.governing:.2f}")
    print(f"    Method: {scf.method}")

# --- Step 3: Apply SCF to nominal stress to get hotspot stress ---
nominal_stress_range = 50.0  # MPa (from global FE analysis)
governing_scf = scf_ax.governing  # typically axial governs for T/Y joints
hotspot_stress = nominal_stress_range * governing_scf

print(f"\n=== Hotspot Stress ===")
print(f"  Nominal stress range: {nominal_stress_range:.1f} MPa")
print(f"  Governing SCF (axial): {governing_scf:.2f}")
print(f"  Hotspot stress range: {hotspot_stress:.1f} MPa")

# --- Step 4: Hotspot stress extrapolation (from FE read-out points) ---
plate_t = joint.T  # chord thickness at the weld
distances = recommended_readout_distances(plate_t, method="linear")
print(f"\n  Read-out distances: {[f'{d:.1f}mm' for d in distances]}")

# Simulated FE stresses at read-out points (decreasing away from weld)
stress_04t = hotspot_stress * 0.95  # stress at 0.4t
stress_10t = hotspot_stress * 0.78  # stress at 1.0t
hs_result = extrapolate_hotspot_linear(plate_t, stress_04t, stress_10t)
print(f"  Extrapolated hotspot stress: {hs_result.hotspot_stress:.1f} MPa")
print(f"  Stress gradient: {hs_result.gradient:.2f} MPa/mm")

# --- Step 5: Fatigue check with hotspot stress ---
curve = get_curve("DNV-RP-C203:D:seawater_cp")
N_allow = calculate_endurance(curve, hs_result.hotspot_stress)
print(f"\n=== Fatigue Check ===")
print(f"  S-N curve: {curve.curve_id}")
print(f"  Hotspot stress range: {hs_result.hotspot_stress:.1f} MPa")
print(f"  Allowable cycles: N = {N_allow:.2e}")

In [ ]:
"""Cell 7: Interactive Multi-Standard S-N Curve Explorer.

A comprehensive Plotly chart showing curves from multiple standards,
with dropdown filters for standard and environment.
"""

# Build a rich multi-curve plot — one trace per curve family
families = [
    ("DNV-RP-C203", ["B1", "C", "D", "E", "F", "F3", "W3"], "air"),
    ("BS 7608",     ["C", "D", "E", "F", "F2", "G", "W"],   "air"),
]

stress_ranges = np.logspace(np.log10(5), np.log10(500), 300)

# Color palettes per standard
palettes = {
    "DNV-RP-C203": ["#1b4f72", "#2e86c1", "#3498db", "#5dade2", "#85c1e9", "#aed6f1", "#d4e6f1"],
    "BS 7608":     ["#78281f", "#b03a2e", "#cb4335", "#e74c3c", "#ec7063", "#f1948a", "#f5b7b1"],
}

fig_all = go.Figure()

for std, classes, env in families:
    palette = palettes[std]
    for i, cls in enumerate(classes):
        cid = f"{std}:{cls}:{env}"
        try:
            c = get_curve(cid)
            N = calculate_endurance(c, stress_ranges)
            fig_all.add_trace(go.Scatter(
                x=N, y=stress_ranges,
                mode="lines",
                name=f"{std} {cls}",
                line=dict(width=2, color=palette[i % len(palette)]),
                legendgroup=std,
                legendgrouptitle_text=std,
                hovertemplate=(
                    f"<b>{std} Class {cls}</b><br>"
                    "N = %{x:.2e}<br>S = %{y:.1f} MPa<extra></extra>"
                ),
            ))
        except KeyError:
            pass  # curve not in library — skip silently

fig_all.update_layout(
    title="S-N Curve Explorer — DNV-RP-C203 vs BS 7608 (Air)",
    xaxis=dict(
        title="Cycles to Failure (N)", type="log",
        range=[3, 10],  # 1e3 to 1e10
        showgrid=True, gridcolor="#f0f0f0", minor=dict(showgrid=True, gridcolor="#f8f8f8"),
    ),
    yaxis=dict(
        title="Stress Range (MPa)", type="log",
        range=[np.log10(5), np.log10(500)],
        showgrid=True, gridcolor="#f0f0f0",
    ),
    template="plotly_white",
    width=1000, height=650,
    legend=dict(
        groupclick="togglegroup",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#ccc", borderwidth=1,
    ),
    annotations=[dict(
        text="Click legend groups to toggle standards",
        xref="paper", yref="paper", x=0.5, y=-0.12,
        showarrow=False, font=dict(size=11, color="#888"),
    )],
)
fig_all.show()

print(f"Plotted {len(fig_all.data)} S-N curves from 2 standards.")

## What This Means for Your Project

The `digitalmodel` fatigue toolkit automates the engineering workflow that traditionally takes days of manual spreadsheet work:

| Manual Process | Automated with digitalmodel |
|---|---|
| Look up S-N curve parameters from standard tables | `get_curve("DNV-RP-C203:D:air")` — instant, auditable |
| Calculate SCFs from Efthymiou equations by hand | `efthymiou_ty_axial(joint)` — validated, unit-tested |
| Extrapolate hotspot stresses from FE results | `extrapolate_hotspot_linear(t, s1, s2)` — per DNV-RP-C203 |
| Miner's rule summation across stress bins | Vectorised `calculate_endurance()` + numpy — seconds, not hours |
| Compare curves across standards for code compliance | `compare_curves([...])` — side-by-side in one line |

### Capabilities

- **Automated fatigue screening** across multiple standards overnight
- **Environment-specific curve selection** per DNV-RP-C203 guidance
- **Full audit trail** from stress range to damage to remaining life
- **Parametric studies** — vary geometry, environment, or standard in a loop
- **Report-ready outputs** — Plotly charts export to HTML, PDF, or PowerPoint

### Integration

```python
# pip install digitalmodel
from digitalmodel.fatigue.sn_library_api import list_curves, get_curve, calculate_endurance
```

Works with your existing FE post-processing pipeline (ANSYS, Abaqus, OrcaFlex) — feed stress ranges in, get fatigue lives out.

---

**Contact**: info@aceengineer.com | [aceengineer.com](https://aceengineer.com)